# Rice Classification CNN Training

This notebook trains a Convolutional Neural Network (CNN) to classify images of rice grains into 5 different varieties. The model uses data augmentation and regularization techniques to prevent overfitting and achieve high accuracy on the validation set.

## Dataset
- **Source**: Rice Image Dataset from Murat Koklu's datasets
- **Classes**: 5 different rice varieties
- **Total Images**: 75,000 images
- **Training/Validation Split**: 80/20 split (60,000 training, 15,000 validation)
- **Image Size**: 224x224 pixels (RGB)

## Model Architecture
- Data augmentation layer (random flips and rotations)
- 2 Convolutional layers with MaxPooling
- Fully connected layers with dropout and L2 regularization
- Softmax output for multi-class classification

## Training Configuration
- **Epochs**: 5
- **Batch Size**: 16
- **Optimizer**: Adam
- **Loss Function**: Categorical Cross-entropy
- **Regularization**: L2 regularization (λ=0.01) + Dropout (20%)


In [ ]:
import os
# Turn off TensorFlow info messages to reduce console output
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'

import tensorflow as tf
from keras import models, layers, regularizers
import matplotlib.pyplot as plt
import numpy as np

## Configuration Parameters

All hyperparameters and configuration settings are defined here for easy modification and experimentation.

In [ ]:
# Data preprocessing parameters
NUM_PIXELS = 224
IMAGE_SIZE = (NUM_PIXELS, NUM_PIXELS)
VALIDATION_SPLIT = 0.2
SHUFFLING_SEED = 123  # Fixed seed ensures consistent train/validation splits across runs

# Training parameters
BATCH_SIZE = 16
NUM_EPOCHS = 5

# CNN architecture parameters
KERNEL_SIZE = (4,4)  # Larger kernel size to capture rice grain features effectively
POOL_SIZE = (4,4)  # Larger pooling to reduce memory usage and feature map size

# Regularization parameters to prevent overfitting
DROPOUT_RATE = 0.2
LAMBDA_2 = 0.01

## Dataset Loading Functions

Functions to load and preprocess the rice image dataset.

In [ ]:
def build_dataset(data_dir, subset):
    """
    Create a TensorFlow dataset from image directory.

    Args:
        data_dir (str): Path to directory containing class subdirectories
        subset (str): Either 'training' or 'validation'

    Returns:
        tf.data.Dataset: Dataset containing (image, label) pairs

    Note:
        - Images are automatically resized to IMAGE_SIZE
        - Labels are one-hot encoded (categorical)
        - Initial batch_size=1 allows for flexible rebatching later
    """
    return tf.keras.preprocessing.image_dataset_from_directory(
        data_dir,
        validation_split=VALIDATION_SPLIT,
        subset=subset,
        label_mode="categorical",  # One-hot encoded labels for multi-class classification
        seed=SHUFFLING_SEED,
        image_size=IMAGE_SIZE,
        batch_size=1  # Will be rebatched later with desired batch size
    )

## Model Checkpointing

Function to set up model checkpointing for saving training progress.

In [ ]:
def set_new_checkpoint_callback(checkpoint_dir):
    """
    Creates a ModelCheckpoint callback for saving model weights during training.

    Args:
        checkpoint_dir (str): Directory path where checkpoints will be saved.
                              Each checkpoint will be saved as 'ckpt_{epoch}' where
                              epoch is automatically set during training.

    Returns:
        tf.keras.callbacks.ModelCheckpoint: Configured checkpoint callback that saves
                                            weights only at the end of each epoch.
    """
    # The {epoch} placeholder is automatically formatted by ModelCheckpoint
    # when the callback is used in model.fit(). The fit() method passes the
    # current epoch number to the callback, which substitutes it into the filename.
    checkpoint_prefix = os.path.join(checkpoint_dir, 'ckpt_{epoch}')
    checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_prefix,
        save_weights_only=True
    )
    return checkpoint_callback

## Model Architecture

Function to build the CNN model with data augmentation and regularization.

In [ ]:
def prepare_new_model(image_size, num_output_neurons):
    """
    Build and compile a CNN model for rice classification.

    Args:
        image_size (tuple): Input image dimensions (height, width)
        num_output_neurons (int): Number of classes to predict

    Returns:
        tf.keras.Model: Compiled CNN model ready for training

    Architecture:
        1. Data Augmentation Layer (reduces overfitting)
        2. Conv2D (32 filters) + MaxPooling2D
        3. Conv2D (64 filters) + MaxPooling2D
        4. Flatten + Dense (128 units with L2 reg) + Dropout
        5. Output Dense layer with softmax activation
    """
    data_augmentation = models.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.2),  # Random rotation up to ±20% (±72 degrees)
    ])

    model = models.Sequential([
        data_augmentation,

        layers.Conv2D(32, kernel_size=KERNEL_SIZE, activation='relu',
                     input_shape=image_size+(3,)),  # 3 channels for RGB
        layers.MaxPooling2D(pool_size=POOL_SIZE),

        layers.Conv2D(64, kernel_size=KERNEL_SIZE, activation='relu'),
        layers.MaxPooling2D(pool_size=POOL_SIZE),

        layers.Flatten(),

        layers.Dense(128, activation='relu',
                    kernel_regularizer=regularizers.L2(LAMBDA_2)),
        layers.Dropout(DROPOUT_RATE),

        layers.Dense(num_output_neurons, activation='softmax')
    ])

    model.compile(
        loss='categorical_crossentropy',  # Standard loss for multi-class classification
        optimizer='adam',  # Optimizer uses default learning rate
        metrics=['accuracy']
    )

    return model

## Dataset Download and Preparation

Download the rice image dataset and prepare it for training.

In [ ]:
tf.keras.utils.get_file(
    origin='https://www.muratkoklu.com/datasets/vtdhnd09.php',
    extract=True
)

229550800/229550800 [==============================] - 10s 0us/step


'/root/.keras/datasets/vtdhnd09.php'

In [7]:
# Verify the dataset was extracted successfully
!ls /root/.keras/datasets/

Rice_Image_Dataset  vtdhnd09.php


In [ ]:
data_dir = "/root/.keras/datasets/Rice_Image_Dataset"

## Dataset Loading and Preprocessing

Load the training and validation datasets, configure batching, and extract metadata.

In [ ]:
train_ds = build_dataset(data_dir, "training")

# Extract metadata before rebatching (cardinality changes after batching)
train_ds_size = train_ds.cardinality().numpy()
class_names = tuple(train_ds.class_names)

# build_dataset() returns data with a batch size of 1, so unbatch first before
# rebatching with desired batch size.
train_ds = train_ds.unbatch().batch(BATCH_SIZE)
train_ds = train_ds.repeat()

val_ds = build_dataset(data_dir, "validation")
val_ds_size = val_ds.cardinality().numpy()
val_ds = val_ds.unbatch().batch(BATCH_SIZE)

Found 75000 files belonging to 5 classes.
Using 60000 files for training.
Found 75000 files belonging to 5 classes.
Using 15000 files for validation.


## Model Initialization

Create the CNN model with the appropriate number of output classes.

In [ ]:
model = prepare_new_model(IMAGE_SIZE, len(class_names))

## Training Configuration

Calculate training steps and set up model checkpointing.

In [ ]:
steps_per_epoch = train_ds_size // BATCH_SIZE  # E.g., 60,000 / 16 = 3,750 steps
validation_steps = val_ds_size // BATCH_SIZE   # E.g., 15,000 / 16 = 937 steps

In [12]:
# Training iteration number for checkpoint organization
# Increment this for each new training run to avoid overwriting checkpoints
current_iteration = 2

In [ ]:
# Saves model weights after each epoch for recovery and analysis
checkpoint_callback = set_new_checkpoint_callback(
    checkpoint_dir=f'drive/MyDrive/Colab Notebooks/RiceML/training_checkpoints_{current_iteration}'
)

## Model Training

Train the CNN model with the configured parameters and monitoring.

In [ ]:
history = model.fit(
    train_ds,
    epochs=NUM_EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_ds,
    validation_steps=validation_steps,
    callbacks=[checkpoint_callback]
)

Epoch 1/5
3750/3750 [==============================] - 208s 53ms/step - loss: 0.7319 - accuracy: 0.9361 - val_loss: 0.2313 - val_accuracy: 0.9724
Epoch 2/5
3750/3750 [==============================] - 194s 52ms/step - loss: 0.2392 - accuracy: 0.9637 - val_loss: 0.1742 - val_accuracy: 0.9783
Epoch 3/5
3750/3750 [==============================] - 192s 51ms/step - loss: 0.2140 - accuracy: 0.9671 - val_loss: 0.1724 - val_accuracy: 0.9789
Epoch 4/5
3750/3750 [==============================] - 195s 52ms/step - loss: 0.1906 - accuracy: 0.9693 - val_loss: 0.1597 - val_accuracy: 0.9799
Epoch 5/5
3750/3750 [==============================] - 193s 51ms/step - loss: 0.1792 - accuracy: 0.9709 - val_loss: 0.1613 - val_accuracy: 0.9803


## Training Results Summary

- **Final Training Accuracy**: 97.09%
- **Final Validation Accuracy**: 98.03%
- **Final Training Loss**: 0.1792
- **Final Validation Loss**: 0.1613

### Next Steps:
- Evaluate model on test set
- Analyze per-class performance
- Consider model deployment options
- Experiment with different architectures or hyperparameters if needed

In [ ]:
# Additional analysis and visualization code can be added here
# For example:
# - Plot training history
# - Generate confusion matrix
# - Visualize model predictions on sample images
# - Save the final model for deployment